In [0]:
  # Top cell of the test notebook
%pip install -e /Workspace/Users/freddielutu@gmail.com/Colibri_Tech_Task
dbutils.library.restartPython()

In [0]:
 import os, sys

# Workspace filesystem can't create __pycache__ — must be set before pytest imports.
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True

# Make wind_pipeline importable even if the editable wheel install didn't land.
src_path = "/Workspace/Users/freddielutu@gmail.com/Colibri_Tech_Task/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pytest

exit_code = pytest.main([
    "/Workspace/Users/freddielutu@gmail.com/Colibri_Tech_Task/tests/unit/test_bronze_clean.py",
    "-v",
    "--tb=short",   
    "-p", "no:cacheprovider",
])
print(f"\nexit code: {exit_code}")

In [0]:
import os, sys

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
src_path = "/Workspace/Users/freddielutu@gmail.com/Colibri_Tech_Task/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pytest
exit_code = pytest.main([
    "/Workspace/Users/freddielutu@gmail.com/Colibri_Tech_Task/tests/unit/test_gold_summary.py",
    "-v", "--tb=short", "-p", "no:cacheprovider",
])
print(f"\nexit code: {exit_code}")

In [0]:
import os, sys
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
src_path = "/Workspace/Users/freddielutu@gmail.com/Colibri_Tech_Task/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pytest
exit_code = pytest.main([
    "/Workspace/Users/freddielutu@gmail.com/Colibri_Tech_Task/tests/unit/test_gold_anomalies.py",
    "-v", "--tb=short", "-p", "no:cacheprovider",
])
print(f"\nexit code: {exit_code}")

In [0]:
import os, sys
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
src_path = "/Workspace/Users/freddielutu@gmail.com/Colibri_Tech_Task/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pytest
exit_code = pytest.main([
    "/Workspace/Users/freddielutu@gmail.com/Colibri_Tech_Task/tests/unit",
    "-v", "--tb=short", "-p", "no:cacheprovider",
])
print(f"\nexit code: {exit_code}")

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS wind_dev;
CREATE SCHEMA  IF NOT EXISTS wind_dev.raw;
CREATE SCHEMA  IF NOT EXISTS wind_dev.silver;
CREATE SCHEMA  IF NOT EXISTS wind_dev.gold;
CREATE VOLUME  IF NOT EXISTS wind_dev.raw.turbines;

In [0]:
%sql
LIST '/Volumes/wind_dev/raw/turbines/'

In [0]:
import os, sys
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
src = "/Workspace/Users/freddielutu@gmail.com/Colibri_Tech_Task/src"
if src not in sys.path:
    sys.path.insert(0, src)

from wind_pipeline.entrypoints import ingest_clean, summarise, detect_anomalies

# Bronze + silver: read CSVs, clean, write wind_dev.silver.power_readings
ingest_clean([
    "--source-volume", "/Volumes/wind_dev/raw/turbines",
    "--catalog", "wind_dev",
])

# Gold summary: write wind_dev.gold.daily_summary
summarise(["--catalog", "wind_dev"])

# Gold anomalies: write wind_dev.gold.daily_anomalies
detect_anomalies([
    "--catalog", "wind_dev",
    "--std-threshold", "2.0",
])

In [0]:
%sql
-- 1) Sanity: cleaned readings count + impossible values purged
SELECT COUNT(*) AS rows,
        MIN(power_output) AS min_p,
        MAX(power_output) AS max_p,
        SUM(CASE WHEN power_output < 0 THEN 1 ELSE 0 END) AS negatives
FROM wind_dev.silver.power_readings;


In [0]:
%sql
-- 2) Summary stats per turbine for the first day
SELECT turbine_id, period_date,
        ROUND(min_power_output, 3) AS min_mw,
        ROUND(max_power_output, 3) AS max_mw,
        ROUND(avg_power_output, 3) AS avg_mw,
        sample_count
FROM wind_dev.gold.daily_summary
WHERE period_date = (SELECT MIN(period_date) FROM wind_dev.gold.daily_summary)
ORDER BY turbine_id;

In [0]:
%sql
-- 3) Top 5 worst anomalies across the whole month
SELECT turbine_id, period_date,
        ROUND(avg_power_output, 3) AS avg_mw,
        ROUND(fleet_mean_power, 3) AS fleet_mean,
        ROUND(z_score, 2)          AS z
FROM wind_dev.gold.daily_anomalies
WHERE is_anomaly = true
ORDER BY ABS(z_score) DESC
LIMIT 5;

In [0]:
%sql
-- 4) Anomalies per day
SELECT period_date, COUNT(*) AS flagged
FROM wind_dev.gold.daily_anomalies
WHERE is_anomaly = true
GROUP BY period_date
ORDER BY period_date;